In [3]:
import polars as pl
df_255 = pl.read_csv(
    "datasets/dataset_merged.csv",
)



In [4]:
import re
# Add sub_sequence column (21 aa from 'position')


WINDOW = 255
HALF = WINDOW // 2  # 256


def get_fragments_from_struct(row_dict: dict) -> dict:
    """
    Funkce extrahuje fragmenty o velikosti WINDOW.
    Přijímá slovník a vrací slovník s novými hodnotami.
    """
    mut_type = row_dict["mut_type"]
    original_seq = row_dict["original_seq_full"]
    mutated_seq = row_dict["mutated_seq_full"]

    # Získání první číselné hodnoty ze sloupce 'mut_type'
    match = re.search(r'\d+', mut_type)
    if not match:
        return {"fragment_255_mut": None, "fragment_255_org": None}

    # Pozice je 1-based, pro index v Pythonu odečteme 1
    center_index = int(match.group(0)) - 1
    seq_len = len(original_seq)

    # Výpočet počátečního a koncového indexu
    start = center_index - HALF
    end = start + WINDOW

    # Ošetření okrajů sekvence
    if start < 0:
        start = 0
        end = WINDOW
    if end > seq_len:
        end = seq_len
        start = seq_len - WINDOW
    if start < 0: # Zajištění pro sekvence kratší než WINDOW
        start = 0

    # Extrakce fragmentů
    org_fragment = original_seq[start:end]
    mut_fragment = mutated_seq[start:end]

    # Funkce musí vrátit slovník s názvy budoucích sloupců
    return {"fragment_255_mut": mut_fragment, "fragment_255_org": org_fragment}

# Aplikace funkce na DataFrame
df_255 = df_255.with_columns(
    # 1. Seskupíme potřebné sloupce do dočasné struktury
    pl.struct(["mut_type", "original_seq_full", "mutated_seq_full"])
    # 2. Aplikujeme funkci na tuto strukturu.
    #    Je nutné specifikovat návratový typ pro optimalizaci.
    .map_elements(
        get_fragments_from_struct,
        return_dtype=pl.Struct([
            pl.Field("fragment_255_mut", pl.String),
            pl.Field("fragment_255_org", pl.String),
        ])
    )
    # 3. Dáme výsledné struktuře dočasný název
    .alias("fragments_struct")
).unnest("fragments_struct") # 4. Rozbalíme strukturu do finálních sloupců



df_255

original_seq_full,mutated_seq_full,mut_type,target,reverse,data_source,fragment_255_mut,fragment_255_org
str,str,str,f64,bool,str,str,str
"""SAGGTYTWNTKEEAKQAFKELLKEKRVPSN…","""SAGGTYTWNTKEEAKQAFKELLKEKPVPSN…","""R22P""",-0.209184,false,"""megascale""","""SAGGTYTWNTKEEAKQAFKELLKEKPVPSN…","""SAGGTYTWNTKEEAKQAFKELLKEKRVPSN…"
"""SAGGSAGGSAGGHEITLHINGRRVKLRFRD…","""SAGGSAGGSAGGHEITLHINGRRVKLRFTD…","""R17T""",-0.110833,false,"""megascale""","""SAGGSAGGSAGGHEITLHINGRRVKLRFTD…","""SAGGSAGGSAGGHEITLHINGRRVKLRFRD…"
"""SAGGSAGGSKDPKFEAAYDFPGSGSSSELP…","""SAGGSAGGSKDPKFEAAMDFPGSGSSSELP…","""Y9M:K24Q""",-0.151356,false,"""megascale""","""SAGGSAGGSKDPKFEAAMDFPGSGSSSELP…","""SAGGSAGGSKDPKFEAAYDFPGSGSSSELP…"
"""SAGGNKASVVANQLIPINTALTLIMMKAEV…","""SAGGNKASVVANQLIPINTALTLIMMKAEV…","""K39N""",-0.038819,false,"""megascale""","""SAGGNKASVVANQLIPINTALTLIMMKAEV…","""SAGGNKASVVANQLIPINTALTLIMMKAEV…"
"""SAGGSAGWVPTKREEKYGVAFYNYDARGAD…","""SAGGSAGWVPTKREEKYGVAFYNYDARGAD…","""D31M:T47G""",-0.187832,false,"""megascale""","""SAGGSAGWVPTKREEKYGVAFYNYDARGAD…","""SAGGSAGWVPTKREEKYGVAFYNYDARGAD…"
…,…,…,…,…,…,…,…
"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""V1059C""",-0.45026,false,"""lehner""","""FGFGASIFSQASNLISTAGQPGPHSQSGPG…","""FGFGASIFSQASNLISTAGQPGPHSQSGPG…"
"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""W1059C""",-0.374472,false,"""lehner""","""FGFGASIFSQASNLISTAGQPGPHSQSGPG…","""FGFGASIFSQASNLISTAGQPGPHSQSGPG…"
"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""Y1059C""",-0.462593,false,"""lehner""","""FGFGASIFSQASNLISTAGQPGPHSQSGPG…","""FGFGASIFSQASNLISTAGQPGPHSQSGPG…"


In [5]:
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1
HOLDOUT_RATIO = VAL_RATIO + TEST_RATIO # ~20% pro validaci a testování

# Získání celkového počtu řádků
n_rows = len(df_255)
print(f"Celkový počet řádků v datasetu: {n_rows}")

# 1. Spočítáme frekvenci každé unikátní sekvence
sequence_counts = df_255.group_by("original_seq_full").len().rename({"len": "count"})

# 2. Seřadíme sekvence od nejčastějších po nejméně časté
sequences_sorted_by_freq = sequence_counts.sort("count", descending=True)

# Získáme seřazený seznam názvů sekvencí
sorted_unique_sequences = sequences_sorted_by_freq.get_column("original_seq_full").to_list()

print("\nNejčastěji zastoupené sekvence:")
print(sequences_sorted_by_freq.head(5))


# 3. Postupný výběr sekvencí pro holdout sadu (validace + test)
holdout_sequences = []
current_holdout_size = 0
target_holdout_size = int(HOLDOUT_RATIO * n_rows)

# Iterujeme přes SEŘAZENÝ seznam (od nejčastějších)
for seq in sorted_unique_sequences:
    # Přidáme další sekvenci do seznamu pro holdout
    holdout_sequences.append(seq)

    # Zjistíme, kolik řádků v datasetu odpovídá dosud vybraným sekvencím
    size_so_far = df_255.filter(pl.col("original_seq_full").is_in(holdout_sequences)).height

    # Pokud jsme dosáhli nebo překročili cílovou velikost, ukončíme cyklus
    if size_so_far >= target_holdout_size:
        break

print(f"\nVybráno {len(holdout_sequences)} nejčastějších unikátních sekvencí pro validaci a testování.")

# 4. Rozdělení datasetu na trénovací a "holdout" část
train_df = df_255.filter(
    ~pl.col("original_seq_full").is_in(holdout_sequences)
)

holdout_pool_df = df_255.filter(
    pl.col("original_seq_full").is_in(holdout_sequences)
)

# 5. Rozdělení "holdout" části na validační a testovací sady
# Zamícháme data, aby bylo rozdělení náhodné
holdout_pool_df = holdout_pool_df.sample(fraction=1, shuffle=True, seed=123)

# Velikost validační sady je 10 % z *původního celkového počtu*
val_size = int(VAL_RATIO * n_rows)

val_df = holdout_pool_df.slice(0, val_size)
# Testovací sada je zbytek z holdout dat
test_df = holdout_pool_df.slice(val_size)

# (Volitelné) Vytvoření testovacího setu bez reverzních mutací
test_df_noreverse = test_df.filter(pl.col("reverse") == False)

# --- Konec úprav ---

# Definice prefixu pro názvy souborů
DATASET_NAME_PERFIX = "datasets/dataset_255w_"

# Uložení každého datasetu do samostatného CSV souboru
train_df.write_csv(f"{DATASET_NAME_PERFIX}train.csv")
val_df.write_csv(f"{DATASET_NAME_PERFIX}validation.csv")
test_df.write_csv(f"{DATASET_NAME_PERFIX}test.csv")
test_df_noreverse.write_csv(f"{DATASET_NAME_PERFIX}noreverse_test.csv")

# Výpis finálních statistik
print("\n--- Výsledky rozdělení ---")
print(f"Trénovací sada:   {len(train_df):>6} řádků ({len(train_df)/n_rows:>6.1%})")
print(f"Validační sada:    {len(val_df):>6} řádků ({len(val_df)/n_rows:>6.1%})")
print(f"Testovací sada:     {len(test_df):>6} řádků ({len(test_df)/n_rows:>6.1%})")
print("-------------------------------")
print(f"Celkem zpracováno: {len(train_df) + len(val_df) + len(test_df):>6} řádků")
print(f"\nSoubory byly úspěšně uloženy s prefixem '{DATASET_NAME_PERFIX}'.")

Celkový počet řádků v datasetu: 1949832

Nejčastěji zastoupené sekvence:
shape: (5, 2)
┌─────────────────────────────────┬───────┐
│ original_seq_full               ┆ count │
│ ---                             ┆ ---   │
│ str                             ┆ u32   │
╞═════════════════════════════════╪═══════╡
│ MLEAIDKNRALHAAERLQTKLRERGDVANE… ┆ 7077  │
│ MAERGGDGGESERFNPGELRMAQQQALRFR… ┆ 5734  │
│ MSKSLKKKSHWTSKVHESVIGRNPEGQLGF… ┆ 5505  │
│ MDCLCIVTTKKYRYQDEDTPPLEHSPAHLP… ┆ 5396  │
│ MRPGTGAERGGLMVSEMESHPPSQGPGDGE… ┆ 5083  │
└─────────────────────────────────┴───────┘

Vybráno 194 nejčastějších unikátních sekvencí pro validaci a testování.

--- Výsledky rozdělení ---
Trénovací sada:   1558817 řádků ( 79.9%)
Validační sada:    194983 řádků ( 10.0%)
Testovací sada:     196032 řádků ( 10.1%)
-------------------------------
Celkem zpracováno: 1949832 řádků

Soubory byly úspěšně uloženy s prefixem 'datasets/dataset_255w_'.


In [4]:
train_df

original_seq_full,mutated_seq_full,mut_type,target,reverse,data_source,fragment_255_mut,fragment_255_org
str,str,str,f64,bool,str,str,str
"""SAGGSAGGSAGGHEITLHINGRRVKLRFRD…","""SAGGSAGGSAGGHEITLHINGRRVKLRFTD…","""R17T""",-0.110833,false,"""megascale""","""SAGGSAGGSAGGHEITLHINGRRVKLRFTD…","""SAGGSAGGSAGGHEITLHINGRRVKLRFRD…"
"""SAGGNKASVVANQLIPINTALTLIMMKAEV…","""SAGGNKASVVANQLIPINTALTLIMMKAEV…","""K39N""",-0.038819,false,"""megascale""","""SAGGNKASVVANQLIPINTALTLIMMKAEV…","""SAGGNKASVVANQLIPINTALTLIMMKAEV…"
"""SENVVSAPMPGKVLRVLVRVGDRVRVGQGL…","""SENVVSAPMPGKVLRVLVRVGDRVRVPQGL…","""G26P""",-0.384118,false,"""megascale""","""SENVVSAPMPGKVLRVLVRVGDRVRVPQGL…","""SENVVSAPMPGKVLRVLVRVGDRVRVGQGL…"
"""LQLFIKTLTGKTFTVEMEPSDTIENLKAKI…","""LQLFIKTLTGKTFTVEMEPSDTIENLKAKI…","""Q49L""",-0.36507,false,"""megascale""","""LQLFIKTLTGKTFTVEMEPSDTIENLKAKI…","""LQLFIKTLTGKTFTVEMEPSDTIENLKAKI…"
"""SAGGSAGGSAGGSNSLAEAKVLANRELDKY…","""SAGGSAGGSAGGSNSLAEAKVLANRELDKY…","""I27T""",-0.352598,false,"""megascale""","""SAGGSAGGSAGGSNSLAEAKVLANRELDKY…","""SAGGSAGGSAGGSNSLAEAKVLANRELDKY…"
…,…,…,…,…,…,…,…
"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""V1059C""",-0.45026,false,"""lehner""","""FGFGASIFSQASNLISTAGQPGPHSQSGPG…","""FGFGASIFSQASNLISTAGQPGPHSQSGPG…"
"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""W1059C""",-0.374472,false,"""lehner""","""FGFGASIFSQASNLISTAGQPGPHSQSGPG…","""FGFGASIFSQASNLISTAGQPGPHSQSGPG…"
"""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""MGNEASLEGEGLPEGLAAAAAAGGGASGAG…","""Y1059C""",-0.462593,false,"""lehner""","""FGFGASIFSQASNLISTAGQPGPHSQSGPG…","""FGFGASIFSQASNLISTAGQPGPHSQSGPG…"


In [11]:
df_255.filter(pl.col("original_seq_full").is_in(holdout_sequences))

original_seq_full,mutated_seq_full,mut_type,target,reverse,data_source,fragment_255_mut,fragment_255_org
str,str,str,f64,bool,str,str,str
"""SAGGSIIYNLKLIREKKKISQSELAALLEV…","""SAGGSIINNLKLIREKKKISQSELAALLEV…","""N3Y""",0.313876,true,"""megascale""","""SAGGSIINNLKLIREKKKISQSELAALLEV…","""SAGGSIIYNLKLIREKKKISQSELAALLEV…"
"""SAGGSAGGSAGGKELVLVLYDYQEKSPREV…","""SAGGSAGGSAGGKELVLVLYDYQEKSPREV…","""K21C""",0.015784,true,"""megascale""","""SAGGSAGGSAGGKELVLVLYDYQEKSPREV…","""SAGGSAGGSAGGKELVLVLYDYQEKSPREV…"
"""SAGGMNLTVNGKPSTVDGAESLNVTELLSA…","""SAGGMNLTVNGKPSTVDGAESLNVTELLSA…","""R46T""",0.190525,true,"""megascale""","""SAGGMNLTVNGKPSTVDGAESLNVTELLSA…","""SAGGMNLTVNGKPSTVDGAESLNVTELLSA…"
"""SAGGSAGGSAGGDEVRLHVNGHTGEFRGID…","""SAGGSAGGSAGGDEVRLHVNGHTIEFRGID…","""I12G""",0.147245,true,"""megascale""","""SAGGSAGGSAGGDEVRLHVNGHTIEFRGID…","""SAGGSAGGSAGGDEVRLHVNGHTGEFRGID…"
"""SAGGSAGGSAGGTYYTVKSGDTANKIAAQY…","""SAGGSAGGSAGGTYYTVKSGDTANKIAAQY…","""V20E""",0.210962,true,"""megascale""","""SAGGSAGGSAGGTYYTVKSGDTANKIAAQY…","""SAGGSAGGSAGGTYYTVKSGDTANKIAAQY…"
…,…,…,…,…,…,…,…
"""MDPGAGSETSLTVNEQVIVMSGHETIRVLE…","""MDPGAGSETSLTVNEQVIVMSGHETIRVLE…","""S202K""",-0.359692,false,"""lehner""","""GSSAEATVKSPPGIPPSPATAIATFSQAPS…","""GSSAEATVKSPPGIPPSPATAIATFSQAPS…"
"""MDPGAGSETSLTVNEQVIVMSGHETIRVLE…","""MDPGAGSETSLTVNEQVIVMSGHETIRVLE…","""T202K""",-0.13698,false,"""lehner""","""GSSAEATVKSPPGIPPSPATAIATFSQAPS…","""GSSAEATVKSPPGIPPSPATAIATFSQAPS…"
"""MDPGAGSETSLTVNEQVIVMSGHETIRVLE…","""MDPGAGSETSLTVNEQVIVMSGHETIRVLE…","""V202K""",-0.027627,false,"""lehner""","""GSSAEATVKSPPGIPPSPATAIATFSQAPS…","""GSSAEATVKSPPGIPPSPATAIATFSQAPS…"


In [12]:
holdout_sequences

['...*DIIKILNKKGQQGWWRGEIYGRVGWFPANYVEEDYSEYC',
 '...']